# NIS-seq Error Analysis

**Match-once pipeline** for NIS-seq HeLa IL-1b data (screens E-9 and E-9C).

All 8 spot-data files are loaded and matched to the Brunello library in a single
pass. The matching results (indices, distances, ties) are stored in memory so
that downstream analyses -- intensity threshold sweeps, per-replicate comparisons,
channel matrix estimation -- can be computed without re-matching.

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

# --- experiments/nisseq_error_analysis: parameters from the environment ---
# Parameterized copy of the original analysis notebook.
# Only this cell differs from it: the original cell hard-codes absolute paths (and
# would overwrite the reference matrices). Here every path and
# switch comes from an NISSEQ_* environment variable (run.sh sets them all). Relative
# defaults resolve against this notebook's folder, which is the kernel's cwd under
# nbconvert. error_utils.py sits beside this notebook.
ERROR_UTILS_DIR = Path(os.environ.get('NISSEQ_ERROR_UTILS_DIR', '.'))
sys.path.insert(0, str(ERROR_UTILS_DIR.resolve()))
from error_utils import (
    load_brunello_library, load_spot_data, filter_valid_reads,
    encode_sequences, match_to_library, compute_error_statistics,
    plot_per_position_error_rates, plot_channel_matrix,
    plot_positional_channel_matrices, plot_per_base_error_rates,
    plot_position_channel_error_heatmap,
    plot_position_channel_error_lines,
    plot_per_true_base_error_rates,
    NUM_BASES,
)
import duet
from duet.codebook_evaluator import HammingDistance
from duet.plotting import apply_style

apply_style()

# Data folder holding NIS-seq_HeLa_IL1b/ (the 8 spot files) and Brunello_sgRNAs/.
DATA_DIR = Path(os.environ.get('NISSEQ_DATA_DIR', '../../data/raw/NISseq_HeLa_IL1b'))
LIBRARY_PATH = Path(os.environ.get('NISSEQ_LIBRARY_PATH',
                                   DATA_DIR / 'Brunello_sgRNAs' / 'Brunello_sgRNAs.txt'))
HELA_DIR = DATA_DIR / 'NIS-seq_HeLa_IL1b'

NUM_ROUNDS = 14

# --- Subsample control ---
# NISSEQ_SUBSAMPLE_N = N takes the first N reads of each file (df.head(N) below;
# deterministic, no random sampling). 'none' uses all reads (full analysis; the
# original notebook estimated ~5-7 hours; untested in this experiment).
_subsample_env = os.environ.get('NISSEQ_SUBSAMPLE_N', '50000')
SUBSAMPLE_N = None if _subsample_env.lower() == 'none' else int(_subsample_env)

# Suffix appended to every artifact (noise-model arrays AND saved figures)
# when running in subsample mode, so test runs never overwrite production files.
SUFFIX = f'_subsample{SUBSAMPLE_N}' if SUBSAMPLE_N is not None else ''

# Output directories: one folder per subsample size, never the tracked
# scripts/benchmark/noise_model_matrices/ (the guard below refuses it).
OUTDIR = Path(os.environ.get(
    'NISSEQ_OUTDIR',
    f"../../results/experiments/nisseq_error_analysis/{SUFFIX.lstrip('_') or 'full'}"))
NOISE_MODELS_DIR = OUTDIR / 'noise_matrices'
FIGURES_DIR = OUTDIR / 'figures'
if 'noise_model_matrices' in OUTDIR.resolve().parts:
    raise ValueError(f'Refusing to write into the tracked noise-model folder: {OUTDIR.resolve()}')

# --- Output control ---
# Independent gates so that figure iteration and noise-model export can proceed
# at different cadences. Both default to on here: the outputs go under OUTDIR,
# not to the tracked matrices (the original notebook defaults SAVE_NOISE_MODEL to False).
SAVE_NOISE_MODEL = os.environ.get('NISSEQ_SAVE_NOISE_MODEL', '1') == '1' # gates writes to NOISE_MODELS_DIR
SAVE_FIGURES = os.environ.get('NISSEQ_SAVE_FIGURES', '1') == '1' # gates SVG writes to FIGURES_DIR

print(f"duet:    {Path(duet.__file__).resolve().parent}  (python {sys.executable})")
print(f"inputs:  {HELA_DIR.resolve()}  +  {LIBRARY_PATH.resolve()}")
print(f"outputs: {OUTDIR.resolve()}")

# --- Threshold sweep ---
# Percentile-based thresholds applied per-screen (E-9 and E-9C have different
# intensity distributions, so absolute thresholds would bias the analysis).
THRESHOLD_PERCENTILES = [0, 25, 50, 60, 70, 80, 90]

FILES = {
    ('E-9', 'B2'): 'E-9_B2_seq_combined.txt',
    ('E-9', 'B3'): 'E-9_B3_seq_combined.txt',
    ('E-9', 'B4'): 'E-9_B4_seq_combined.txt',
    ('E-9', 'B5'): 'E-9_B5_seq_combined.txt',
    ('E-9C', 'B2'): 'E-9C_Sequences_B2.txt',
    ('E-9C', 'B3'): 'E-9C_Sequences_B3.txt',
    ('E-9C', 'B4'): 'E-9C_Sequences_B4.txt',
    ('E-9C', 'B5'): 'E-9C_Sequences_B5.txt',
}

if SUBSAMPLE_N is not None:
    print(f"⚠ SUBSAMPLE MODE: {SUBSAMPLE_N:,} reads per file (suffix = '{SUFFIX}')")
else:
    print("Full dataset mode (all reads)")

if not SAVE_NOISE_MODEL:
    print("⚠ SAVE_NOISE_MODEL = False — noise-model .npy files will NOT be written in section 9.")
if not SAVE_FIGURES:
    print("⚠ SAVE_FIGURES = False — figures will NOT be written to FIGURES_DIR.")

## 1. Load Library and Pre-compute BLAS State

In [ ]:
# Load Brunello library and encode reverse complements (first NUM_ROUNDS bases)
library_df = load_brunello_library(LIBRARY_PATH)
rc_seqs = library_df['ReverseComplement'].str[:NUM_ROUNDS].tolist()
library_encoded = encode_sequences(rc_seqs)

print(f"Library size: {len(library_df)} sgRNAs")
print(f"Unique RC-{NUM_ROUNDS}mers: {len(set(rc_seqs))}")
print(f"Encoded shape: {library_encoded.shape}, dtype: {library_encoded.dtype}")

# Precompute BLAS state for matching
hamming = HammingDistance(alphabet_size=NUM_BASES)
precomputed = hamming.precompute_transmitted(library_encoded)

## 2. Match All Reads (Once)

Load all 8 spot-data files and match each to the Brunello library in a single
forward pass. The per-file match results are concatenated into global arrays so
that every downstream analysis reuses the same matching without recomputation.

In [ ]:
REPLICATE_MAP = {'E-9': 0, 'E-9C': 1}
BATCH_MAP = {'B2': 0, 'B3': 1, 'B4': 2, 'B5': 3}

all_encoded_seqs = []
all_match_indices = []
all_best_distances = []
all_second_best_distances = []
all_intensities = []
all_replicates = []
all_batches = []
all_tied_indices = {}

global_offset = 0
first_file = True

for (replicate, batch), filename in tqdm(FILES.items(), desc='Matching files'):
    print(f"\n--- {replicate} {batch}: {filename} ---")
    filepath = HELA_DIR / filename
    df = load_spot_data(filepath)
    df = df.drop(columns=['x', 'y'])
    df = filter_valid_reads(df, seq_length=NUM_ROUNDS)

    # Subsample if requested
    if SUBSAMPLE_N is not None and len(df) > SUBSAMPLE_N:
        df = df.head(SUBSAMPLE_N).copy()
        print(f"  Subsampled to {SUBSAMPLE_N:,} reads")

    n_reads = len(df)
    print(f"  Valid reads: {n_reads:,}")

    encoded = encode_sequences(df['sequence'].tolist())
    result = match_to_library(encoded, precomputed, chunk_size=10000)

    # Sanity check on first file
    if first_file:
        exact_rate = (result['best_distance'] == 0).mean()
        print(f"  Sanity check -- exact match rate (unfiltered): {exact_rate:.1%}")
        # Threshold is low because we match ALL reads without intensity filtering.
        # The original example notebook got ~49% but that was after filtering at
        # intensity >= 1.1. With unfiltered data, many low-intensity noise reads
        # dilute the exact match rate.
        if exact_rate < 0.05:
            raise ValueError(
                f"Exact match rate is {exact_rate:.1%}, expected >= 5%. "
                "Check library / data alignment (reverse complement?)."
            )
        first_file = False

    all_encoded_seqs.append(encoded)
    all_match_indices.append(result['match_index'])
    all_best_distances.append(result['best_distance'])
    all_second_best_distances.append(result['second_best_distance'])
    all_intensities.append(df['intensity'].values.astype(np.float32))
    all_replicates.append(np.full(n_reads, REPLICATE_MAP[replicate], dtype=np.int8))
    all_batches.append(np.full(n_reads, BATCH_MAP[batch], dtype=np.int8))

    # Remap tied_indices keys to global offset
    for local_idx, lib_indices in result['tied_indices'].items():
        all_tied_indices[global_offset + local_idx] = lib_indices

    global_offset += n_reads

    # Free per-file data
    del df, encoded, result

# Concatenate all arrays
all_encoded_seqs = np.concatenate(all_encoded_seqs)
all_match_indices = np.concatenate(all_match_indices)
all_best_distances = np.concatenate(all_best_distances)
all_second_best_distances = np.concatenate(all_second_best_distances)
all_intensities = np.concatenate(all_intensities)
all_replicates = np.concatenate(all_replicates)
all_batches = np.concatenate(all_batches)

combined_match_result = {
    'match_index': all_match_indices,
    'best_distance': all_best_distances,
    'second_best_distance': all_second_best_distances,
    'tied_indices': all_tied_indices,
}

# Summary
n_total = len(all_encoded_seqs)
n_e9 = (all_replicates == 0).sum()
n_e9c = (all_replicates == 1).sum()
n_tied = len(all_tied_indices)
mem_mb = (all_encoded_seqs.nbytes + all_match_indices.nbytes +
          all_best_distances.nbytes + all_second_best_distances.nbytes +
          all_intensities.nbytes) / 1e6

print(f"\n=== Totals ===")
print(f"Total reads:    {n_total:,}")
print(f"  E-9:          {n_e9:,}")
print(f"  E-9C:         {n_e9c:,}")
print(f"Tied reads:     {n_tied:,} ({n_tied/n_total:.1%})")
print(f"Array memory:   ~{mem_mb:.0f} MB")
if SUBSAMPLE_N is not None:
    print(f"\n⚠ Results are from subsampled data ({SUBSAMPLE_N:,} reads/file)")

## 3. Compute Per-Screen Intensity Thresholds

E-9 and E-9C have different intensity distributions (E-9 mean ~2.2, E-9C mean ~1.0).
Using absolute thresholds would disproportionately filter E-9C reads. Instead, we
compute percentile-based thresholds per-screen and create a combined boolean mask
for each percentile level.

In [ ]:
# Compute per-screen percentile thresholds
e9_intensities = all_intensities[all_replicates == 0]
e9c_intensities = all_intensities[all_replicates == 1]

print("Per-screen intensity thresholds:")
print(f"{'Percentile':>10}  {'E-9 threshold':>15}  {'E-9C threshold':>15}")
print("-" * 45)

# For each percentile, create a combined boolean mask that applies
# the screen-specific threshold to each read
percentile_masks = {}
for pct in THRESHOLD_PERCENTILES:
    if pct == 0:
        thresh_e9, thresh_e9c = 0.0, 0.0
    else:
        thresh_e9 = np.percentile(e9_intensities, pct)
        thresh_e9c = np.percentile(e9c_intensities, pct)

    # Apply per-screen thresholds
    mask = np.zeros(len(all_intensities), dtype=bool)
    e9_mask = all_replicates == 0
    e9c_mask = all_replicates == 1
    mask[e9_mask] = all_intensities[e9_mask] >= thresh_e9
    mask[e9c_mask] = all_intensities[e9c_mask] >= thresh_e9c
    percentile_masks[pct] = mask

    print(f"{pct:>10}th  {thresh_e9:>15.4f}  {thresh_e9c:>15.4f}  "
          f"({mask.sum():,} reads pass)")

del e9_intensities, e9c_intensities

## 4. Intensity Threshold Sweep

Compute error statistics at each percentile threshold level.

In [ ]:
sweep_results = {}

for pct in THRESHOLD_PERCENTILES:
    mask = percentile_masks[pct]
    indices = np.where(mask)[0]

    if len(indices) == 0:
        continue

    # Build filtered match result for this threshold
    filtered_match = {
        'match_index': all_match_indices[indices],
        'best_distance': all_best_distances[indices],
        'second_best_distance': all_second_best_distances[indices],
        'tied_indices': {
            new_i: all_tied_indices[orig_i]
            for new_i, orig_i in enumerate(indices)
            if orig_i in all_tied_indices
        },
    }

    stats = compute_error_statistics(
        all_encoded_seqs[indices], library_encoded, filtered_match,
        all_intensities[indices], intensity_threshold=0.0,
    )
    sweep_results[pct] = stats

    exact_rate = (filtered_match['best_distance'] == 0).mean()
    unamb_rate = (filtered_match['best_distance'] < filtered_match['second_best_distance']).mean()
    avg_err = stats.per_position_error_rates.mean()

    print(f"Percentile {pct:>3}th | n_reads {stats.n_reads:>10,} | "
          f"unambiguous {unamb_rate:.1%} | "
          f"exact_match {exact_rate:.1%} | "
          f"avg_error {avg_err:.5f}")

In [ ]:
from duet.plotting import FIGURE_WIDTHS

fig, ax = plt.subplots(figsize=(FIGURE_WIDTHS["half_page"], 2.5))

for pct in THRESHOLD_PERCENTILES:
    if pct not in sweep_results:
        continue
    stats = sweep_results[pct]
    plot_per_position_error_rates(
        stats.per_position_error_rates,
        label=f'{pct}th pctl (n={stats.n_reads:,})',
        ax=ax,
    )

ax.set_title('Per-Position Error Rate by Intensity Percentile Threshold')
all_rates = np.concatenate([sweep_results[p].per_position_error_rates
                            for p in THRESHOLD_PERCENTILES if p in sweep_results])
ax.set_ylim(0, all_rates.max() * 1.1)
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
plt.show()

## 5. Per-Position Error Analysis (Selected Percentile)

In [ ]:
# Select percentile for detailed analysis (adjust based on sweep results above)
SELECTED_PERCENTILE = 50

In [ ]:
stats_pooled = sweep_results[SELECTED_PERCENTILE]
selected_mask = percentile_masks[SELECTED_PERCENTILE]

print(f"Selected percentile: {SELECTED_PERCENTILE}th")
print(f"Reads at threshold: {stats_pooled.n_reads:,}")
print(f"Average error rate: {stats_pooled.per_position_error_rates.mean():.5f}")

In [ ]:
# Per-replicate comparison using selected percentile mask

# --- E-9 ---
e9_mask = (all_replicates == 0) & selected_mask
e9_global = np.where(e9_mask)[0]
e9_local_map = {g: i for i, g in enumerate(e9_global)}
e9_encoded = all_encoded_seqs[e9_global]
e9_match_result = {
    'match_index': all_match_indices[e9_global],
    'best_distance': all_best_distances[e9_global],
    'second_best_distance': all_second_best_distances[e9_global],
    'tied_indices': {
        e9_local_map[g]: all_tied_indices[g]
        for g in e9_global if g in all_tied_indices
    },
}
e9_intensities = all_intensities[e9_global]
stats_e9 = compute_error_statistics(
    e9_encoded, library_encoded, e9_match_result,
    e9_intensities, intensity_threshold=0.0,
)

# --- E-9C ---
e9c_mask = (all_replicates == 1) & selected_mask
e9c_global = np.where(e9c_mask)[0]
e9c_local_map = {g: i for i, g in enumerate(e9c_global)}
e9c_encoded = all_encoded_seqs[e9c_global]
e9c_match_result = {
    'match_index': all_match_indices[e9c_global],
    'best_distance': all_best_distances[e9c_global],
    'second_best_distance': all_second_best_distances[e9c_global],
    'tied_indices': {
        e9c_local_map[g]: all_tied_indices[g]
        for g in e9c_global if g in all_tied_indices
    },
}
e9c_intensities = all_intensities[e9c_global]
stats_e9c = compute_error_statistics(
    e9c_encoded, library_encoded, e9c_match_result,
    e9c_intensities, intensity_threshold=0.0,
)

print(f"E-9  reads: {stats_e9.n_reads:,},  avg error: {stats_e9.per_position_error_rates.mean():.5f}")
print(f"E-9C reads: {stats_e9c.n_reads:,}, avg error: {stats_e9c.per_position_error_rates.mean():.5f}")

# Plot overlay
from duet.plotting import FIGURE_WIDTHS

fig, ax = plt.subplots(figsize=(FIGURE_WIDTHS["half_page"], 2.5))
plot_per_position_error_rates(stats_pooled.per_position_error_rates,
                              label='Pooled', ax=ax, linewidth=2)
plot_per_position_error_rates(stats_e9.per_position_error_rates,
                              label='E-9', ax=ax, linestyle='--')
plot_per_position_error_rates(stats_e9c.per_position_error_rates,
                              label='E-9C', ax=ax, linestyle='--')
ax.set_title(f'Per-Position Error Rate ({SELECTED_PERCENTILE}th percentile threshold)')
all_rates = np.concatenate([stats_pooled.per_position_error_rates,
                            stats_e9.per_position_error_rates,
                            stats_e9c.per_position_error_rates])
ax.set_ylim(0, all_rates.max() * 1.1)
ax.legend()
fig.tight_layout()
plt.show()

## 6. Base-Pair-Level Error Breakdown

In [ ]:
from duet.plotting import FIGURE_WIDTHS

fig, ax = plt.subplots(figsize=(FIGURE_WIDTHS["half_page"], 2.5))
plot_per_base_error_rates(stats_pooled.per_position_base_errors, ax=ax)
ax.set_title(f'Per-Position Error Rates by Base ({SELECTED_PERCENTILE}th percentile threshold)')
fig.tight_layout()
plt.show()

## 7. Channel Transition Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

plot_channel_matrix(stats_pooled.channel_matrix, title='Pooled', ax=axes[0])
plot_channel_matrix(stats_e9.channel_matrix, title='E-9', ax=axes[1])
plot_channel_matrix(stats_e9c.channel_matrix, title='E-9C', ax=axes[2])

fig.suptitle(f'Channel Transition Matrices ({SELECTED_PERCENTILE}th percentile threshold)', fontsize=14)
fig.tight_layout()
plt.show()

# Print pooled matrix values and row sums
print("Pooled channel matrix:")
labels = ['A', 'T', 'C', 'G']
for i, row_label in enumerate(labels):
    row = stats_pooled.channel_matrix[i]
    row_str = '  '.join(f'{v:.6f}' for v in row)
    print(f"  {row_label}: [{row_str}]  sum={row.sum():.6f}")

if SAVE_FIGURES:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    out = FIGURES_DIR / f'channel_matrices_pctl{SELECTED_PERCENTILE}{SUFFIX}.svg'
    fig.savefig(out, bbox_inches='tight')
    print(f'Saved {out}')


## 8. Positional Channel Matrices

In [ ]:
fig = plot_positional_channel_matrices(stats_pooled.positional_channel_matrices)
plt.show()

if SAVE_FIGURES:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    out = FIGURES_DIR / f'positional_channel_grid_pctl{SELECTED_PERCENTILE}{SUFFIX}.svg'
    fig.savefig(out, bbox_inches='tight')
    print(f'Saved {out}')


## 8b. Publication-Quality Error Structure Heatmap

Single-panel view of the `(L, 4, 4)` transition tensor. The 12 off-diagonal
transitions are stacked as rows (grouped by true base), with sequence position
on the x-axis. Horizontal separators demarcate the four true-base blocks. The
top marginal shows the per-position error rate; the right marginal shows the
mean transition probability across positions (colored by true base to reinforce
the block grouping). The colorbar is log-scaled — errors span roughly two
orders of magnitude, and a linear scale collapses the structure.

Reading guide:
- **Vertical stripes** → position-axis structure (certain positions are error-prone)
- **Rows systematically lighter/darker than others** → channel-axis structure
  (certain true bases / transitions are error-prone)
- **Patterns that can't be expressed as row × column outer product** → joint
  structure that motivates the `positional_channel` noise model over uniform
  or pure-positional alternatives.


In [ ]:
fig, axes = plot_position_channel_error_heatmap(
    stats_pooled.positional_channel_matrices,
    per_position_error_rates=stats_pooled.per_position_error_rates,
    cmap='rocket_r',
    log_scale=True,
    title=(
        f'NIS-seq HeLa IL-1b — empirical error structure '
        f'(pooled E-9 + E-9C, {SELECTED_PERCENTILE}th-pctl intensity threshold, '
        f'n={stats_pooled.n_reads:,} reads)'
    ),
)
plt.show()

if SAVE_FIGURES:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    out = FIGURES_DIR / f'error_heatmap_pctl{SELECTED_PERCENTILE}{SUFFIX}.svg'
    fig.savefig(out, bbox_inches='tight')
    print(f'Saved {out}')

## 8c. Grouped Line-Plot View

Alternative single-panel view of the same `(L, 4, 4)` tensor. Same data as 8b,
different encoding:

- **X-axis:** sequence position (1–14)
- **Y-axis:** `P(observed | true)` on a log scale
- **12 lines**, one per off-diagonal transition
  - **Color** = true (source) base (A / T / C / G — ColorBrewer Set1 palette)
  - **Line style + marker** = destination rank within each source
    (solid ● = 1st, dashed ■ = 2nd, dotted ▲ = 3rd; order is ATCG with self removed)
- **End-of-line labels** give the full `X→Y` transition for each line
  (collision-avoided in log space), so identification does not depend on
  decoding the line-style legend.

Trade-offs vs. 8b:
- Line plot makes **position-axis trends** (e.g. errors rising over rounds)
  more obvious and lets readers read off values directly from the y-axis.
- Heatmap makes **channel-axis structure** (systematically higher/lower rows)
  and **non-separability** (joint position×channel structure) more obvious,
  which is the stronger visual argument for the `positional_channel` noise
  model over `positional`.


In [ ]:
fig, ax = plot_position_channel_error_lines(
    stats_pooled.positional_channel_matrices,
    log_scale=True,
)
plt.show()

if SAVE_FIGURES:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    out = FIGURES_DIR / f'error_lines_12_pctl{SELECTED_PERCENTILE}{SUFFIX}.svg'
    fig.savefig(out, bbox_inches='tight')
    print(f'Saved {out}')

## 8d. Simplified 4-Line View (Aggregated by True Base)

Simplification of 8c. Collapses each source's 3 destination lines into a single
per-source error rate:

$$P(\text{error} \mid \text{true} = b) = 1 - P(\text{observed} = b \mid \text{true} = b)$$

- **4 lines**, colored by true base (A/T/C/G, same ColorBrewer Set1 palette as 8b/8c)
- **Log y-axis** for consistency with the other views
- Drops the destination-resolved detail of 8c; keeps the per-source, per-position
  error story in a much more legible form.

Good companion to 8b/8c: this view is for a reader who wants the
headline ("does the error rate vary across positions and across true bases?"),
while 8b/8c provide the destination-resolved supporting detail.


In [ ]:
fig, ax = plot_per_true_base_error_rates(
    stats_pooled.positional_channel_matrices,
    counts=stats_pooled.positional_channel_counts,
    log_scale=False,
)
plt.show()

if SAVE_FIGURES:
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    out = FIGURES_DIR / f'error_lines_4_pctl{SELECTED_PERCENTILE}{SUFFIX}.svg'
    fig.savefig(out, bbox_inches='tight')
    print(f'Saved {out}')

## 9. Export Noise Models

In [ ]:
# 1. Uniform: single average epsilon -> 4x4 matrix
avg_epsilon = stats_pooled.per_position_error_rates.mean()
uniform_matrix = np.full((NUM_BASES, NUM_BASES), avg_epsilon / 3.0)
np.fill_diagonal(uniform_matrix, 1.0 - avg_epsilon)

print("=== Uniform noise model ===")
print(f"  avg_epsilon = {avg_epsilon:.6f}")
print(f"  Matrix:\n{uniform_matrix}\n")

# 2. Positional: per-position error rates (L,)
positional_epsilons = stats_pooled.per_position_error_rates.copy()

print("=== Positional noise model ===")
print(f"  Shape: {positional_epsilons.shape}")
print(f"  Values: {positional_epsilons}\n")

# 3. Channel: aggregated 4x4 transition matrix
channel_matrix = stats_pooled.channel_matrix.copy()

print("=== Channel noise model ===")
print(f"  Shape: {channel_matrix.shape}")
print(f"  Row sums: {channel_matrix.sum(axis=1)}\n")

# 4. Positional channel: (L, 4, 4) transition matrices
positional_channel = stats_pooled.positional_channel_matrices.copy()

print("=== Positional channel noise model ===")
print(f"  Shape: {positional_channel.shape}")
print(f"  Row sums (pos 0): {positional_channel[0].sum(axis=1)}")

In [ ]:
NOISE_MODELS_DIR.mkdir(parents=True, exist_ok=True)

if SUBSAMPLE_N is not None:
    print("⚠ SUBSAMPLE MODE — matrices will be saved with '_subsample' suffix.")
    print("  Re-run with SUBSAMPLE_N = None for production matrices.")

exports = {
    f'nisseq_hela_uniform_pctl{SELECTED_PERCENTILE}{SUFFIX}.npy': uniform_matrix,
    f'nisseq_hela_positional_pctl{SELECTED_PERCENTILE}{SUFFIX}.npy': positional_epsilons,
    f'nisseq_hela_channel_pctl{SELECTED_PERCENTILE}{SUFFIX}.npy': channel_matrix,
    f'nisseq_hela_positional_channel_pctl{SELECTED_PERCENTILE}{SUFFIX}.npy': positional_channel,
}

if not SAVE_NOISE_MODEL:
    print("SAVE_NOISE_MODEL = False — skipping writes. Flip the flag in the first cell to export.")
    for filename, arr in exports.items():
        print(f"  would save {filename}  shape={arr.shape}")
else:
    for filename, arr in exports.items():
        np.save(NOISE_MODELS_DIR / filename, arr)
        print(f"Saved {filename}  shape={arr.shape}")

    # Validate: reload each file and check properties
    print("\n=== Validation ===")
    for filename, original in exports.items():
        loaded = np.load(NOISE_MODELS_DIR / filename)

        # Shape check
        assert loaded.shape == original.shape, f"{filename}: shape mismatch"

        # Non-negativity
        assert np.all(loaded >= 0), f"{filename}: negative values found"

        # Row-stochasticity (for matrices with rows that should sum to 1)
        if loaded.ndim == 2 and loaded.shape[0] == loaded.shape[1]:
            row_sums = loaded.sum(axis=1)
            assert np.allclose(row_sums, 1.0, atol=1e-6), (
                f"{filename}: rows don't sum to 1, got {row_sums}"
            )
            print(f"  {filename}: OK (shape={loaded.shape}, row-stochastic, non-negative)")
        elif loaded.ndim == 3:
            for pos in range(loaded.shape[0]):
                row_sums = loaded[pos].sum(axis=1)
                assert np.allclose(row_sums, 1.0, atol=1e-6), (
                    f"{filename} pos {pos}: rows don't sum to 1"
                )
            print(f"  {filename}: OK (shape={loaded.shape}, all positions row-stochastic, non-negative)")
        else:
            print(f"  {filename}: OK (shape={loaded.shape}, non-negative)")

    print("\nAll validations passed.")


## YAML Config Snippets

Use these snippets in benchmark YAML configs to reference the exported noise models.

**Uniform (as channel matrix):**
```yaml
noise_model:
  type: channel
  channel_matrix_path: scripts/benchmark/noise_model_matrices/channels/nisseq_hela_uniform.npy
```
Note: can also use the native `uniform` type with a scalar epsilon value instead.

**Positional:**
```yaml
noise_model:
  type: positional
  epsilon_path: scripts/benchmark/noise_model_matrices/channels/nisseq_hela_positional.npy
  dissimilarity:
    type: positional_nll
```

**Channel:**
```yaml
noise_model:
  type: channel
  channel_matrix_path: scripts/benchmark/noise_model_matrices/channels/nisseq_hela_channel.npy
```

**Positional channel:**
```yaml
noise_model:
  type: positional_channel
  channel_matrices_path: scripts/benchmark/noise_model_matrices/channels/nisseq_hela_positional_channel.npy
```